# 기본 RAG: 근거를 인용하며 답하기

한국어 질문으로 영어 위키백과 코퍼스(23개 문서)에서 근거를 찾고, 그 근거만 인용해 한국어로 답해요.
먼저 검색만으로 판정하고(6장), 이어서 LLM 답변으로 다시 판정해요(6-2).

## 1. 파싱

`data/raw/{id}.txt`를 `sources.csv` 메타데이터와 함께 읽고, `== 제목 ==` 줄로 섹션을 나눠요.
References, See also 같은 목록 섹션은 빼고, 반도체와 무관한 섹션은 지우지 않고 개수만 세요.
위치는 모두 원문 기준 글자 위치(char_start, char_end)예요.

In [1]:
from fabatlas.corpus import load_corpus, EXCLUDED_SECTIONS

docs = load_corpus()
sections = [s for d in docs for s in d.sections]
off_topic = [s for s in sections if s.off_topic]
print(f"문서 {len(docs)}개, 섹션 {len(sections)}개")
print(f"제외한 섹션 이름: {sorted(EXCLUDED_SECTIONS)}")
print(f"무관 섹션(지우지 않음): {len(off_topic)}개, {sum(len(s.text.split()) for s in off_topic):,}단어")
for s in off_topic[:5]:
    print(f"  {s.doc_id} {s.path}")

문서 23개, 섹션 296개
제외한 섹션 이름: ['Bibliography', 'External links', 'Further reading', 'Notes', 'References', 'Related links', 'See also']
무관 섹션(지우지 않음): 16개, 1,727단어
  S005 Natural occurrence > Geology
  S005 Natural occurrence > Biology
  S005 Uses > Structural use
  S005 Uses > Food, cosmetic, and pharmaceutical applications
  S005 Health effects


## 2. 청킹

문서 전체 본문을 300단어씩 자르고, 다음 청크는 50단어 겹쳐 시작해요(`w300o50`, D-007).
청킹 함수는 `CHUNKERS`에 이름으로 등록해 바꿔 끼울 수 있어요. 나중에 섹션 단위, 500/100과 비교해요.
청크마다 원문 위치를 남겨서, 평가셋 정답을 청킹 방식과 상관없이 적을 수 있어요.

In [2]:
import statistics as st
from fabatlas.chunking import CHUNKERS, chunk_corpus

for name in CHUNKERS:
    ch = chunk_corpus(docs, name)
    w = [len(c.text.split()) for c in ch]
    print(f"{name:9s} 청크 {len(ch):3d}개, 평균 {st.mean(w):5.0f}단어 (최소 {min(w)}, 최대 {max(w)})")

chunks = chunk_corpus(docs, "w300o50")
c = chunks[1]
raw = next(d for d in docs if d.doc_id == c.doc_id).text
print(f"\n예: {c.chunk_id} | {c.section} | 원문 {c.char_start}–{c.char_end}")
print("원문 위치로 꺼낸 앞부분:", raw[c.char_start:c.char_start + 80])

section   청크 296개, 평균   180단어 (최소 11, 최대 1003)
w300o50   청크 219개, 평균   287단어 (최소 51, 최대 300)
w500o100  청크 140개, 평균   463단어 (최소 102, 최대 500)

예: S001§2 | Production > Formation | 원문 1743–3662
원문 위치로 꺼낸 앞부분: for forming crystalline wafers is known as the Czochralski method, invented by P


## 3. 임베딩

BAAI/bge-m3를 커밋 해시로 고정해 쓰고(D-008), 청크마다 1024차원 벡터를 만들어 numpy로 저장해요(D-009).
벡터는 L2 정규화해서 저장해요. 길이가 1이면 내적이 곧 코사인 유사도예요.
같은 청킹이면 저장된 벡터를 불러와서 다시 계산하지 않아요.

In [3]:
import numpy as np
from fabatlas.embedding import MODEL_ID, MODEL_REVISION, build_index, load_model

model = load_model("cpu")
index = build_index("w300o50", "cpu", model=model)
print(f"모델: {MODEL_ID} @ {MODEL_REVISION}")
print(f"벡터: {index.embeddings.shape}, 길이 {np.linalg.norm(index.embeddings, axis=1)[:3].round(3)} …")
print(f"처음 만들 때 CPU 임베딩 시간: {index.meta['embed_seconds']}초 (청크 {index.meta['n_chunks']}개)")

W1018 17:02:02.873000 20155 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Fetching 10 files: 100%|██████████| 10/10 [00:00<00:00, 1999.67it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 52140.43it/s]

모델: BAAI/bge-m3 @ 5617a9f61b028005a4858fdac845db406aefb181
벡터: (219, 1024), 길이 [1. 1. 1.] …
처음 만들 때 CPU 임베딩 시간: 64.7초 (청크 219개)


## 4. 검색

질문도 같은 모델로 벡터를 만든 뒤, 모든 청크 벡터와 내적(= 코사인 유사도)을 한 번에 구해요.
점수가 높은 순서로 top-k(여기서는 5개)를 골라요.

In [4]:
from fabatlas.search import search

hits = search(index, model, "EUV 노광에서 펠리클은 왜 필요한가요?", k=5)
for h in hits:
    print(f"{h.rank}. {h.score:.3f} [{h.chunk.chunk_id}] {h.chunk.title} — {h.chunk.section}")

1. 0.572 [S008§34] EUV lithography — Mask defects > Pellicles
2. 0.554 [S008§33] EUV lithography — Mask defects
3. 0.541 [S008§35] EUV lithography — Mask defects > Pellicles
4. 0.535 [S008§12] EUV lithography — Tool
5. 0.531 [S008§1] EUV lithography — (도입부)


## 5. 답변

검색된 top-k 청크만 근거로 넣고, 문장마다 `[S005§2]` 같은 출처를 달게 하는 프롬프트예요(`fabatlas/prompts.py`).
근거가 부족하면 "코퍼스에서 근거를 찾을 수 없어요"라고만 답하게 해요. 모델은 `gpt-4.1-mini-2025-04-14`, 온도 0이에요.

In [5]:
from fabatlas.prompts import build_messages

msgs = build_messages("EUV 노광에서 펠리클은 왜 필요한가요?", hits[:2])
print(msgs[0]["content"])
print("\n" + msgs[1]["content"][:500] + " …")

당신은 반도체 공정 질문에 답하는 도우미예요.
규칙:
1. 아래 [근거]에 있는 내용만 사용해요. 근거에 없는 사실은 쓰지 않아요.
2. 한국어 해요체로 답해요. 근거는 영어지만 답은 한국어로 써요.
3. 모든 문장 끝에 그 문장의 근거 청크 표시를 달아요. 예: "...이에요. [S005§2]"
4. 근거가 질문에 답하기에 부족하면 다른 말 없이 "코퍼스에서 근거를 찾을 수 없어요."라고만 답해요.

[질문]
EUV 노광에서 펠리클은 왜 필요한가요?

[근거]
[S008§34] EUV lithography — Mask defects > Pellicles
absorption. Particle contamination would be prohibitive if pellicles were not stable above 200 W, i.e., the targeted power for manufacturing.

Heating of the EUV mask pellicle (film temperature up to 750 K for 80 W incident power) is a significant concern, due to the resulting deformation and transmission decrease. ASML developed a 70 nm thick polysilicon pellicle membrane, which allows EUV transmission of 82% …


## 6. 테스트 질문 5개 (검색 기준)

먼저 검색 결과만으로 판정해요. 답변 기준 판정은 6-2에서 해요.
- 통과: top-1이 기대 문서이고, 핵심 근거 문장이 top-3 청크 안에 있음
- 부분: 기대 문서가 top-5 안에 있지만 통과 조건에 못 미침 / 실패: 기대 문서가 top-5 밖 / Q5(답 없음)는 거절이 LLM 단계라 판정 보류

In [6]:
import re
from IPython.display import Markdown

TESTS = [
    ("Q1", "포토 공정에서 감광액(포토레지스트)은 왜 바르나요?", {"S006", "S007"}, r"soluble|developer"),
    ("Q2", "이온 주입을 한 뒤에 왜 어닐링(열처리)을 해야 하나요?", {"S013"}, r"activate dopants"),
    ("Q3", "반응성 이온 식각(RIE)은 왜 습식 식각과 달리 수직으로 깎이나요(이방성)?", {"S011"}, r"negative voltage"),
    ("Q4", "EDS 공정에서는 무엇을 검사하나요?", {"S018"}, r"WPT|WFT"),
    ("Q5", "삼성 3나노 공정의 수율은 얼마인가요?", set(), None),
]

def judge(hits, expected, key):
    if not expected:
        return "판정 보류"
    docs_top5 = [h.chunk.doc_id for h in hits]
    if not set(docs_top5) & expected:
        return "실패"
    key_in_top3 = any(h.chunk.doc_id in expected and re.search(key, h.chunk.text) for h in hits[:3])
    return "통과" if docs_top5[0] in expected and key_in_top3 else "부분"

rows, results = [], {}
for qid, q, expected, key in TESTS:
    hs = search(index, model, q, k=5)
    results[qid] = hs
    first = next((h.rank for h in hs if h.chunk.doc_id in expected), None)
    match = "해당 없음" if not expected else (f"top-1 일치" if first == 1 else f"top-{first}에서 처음 등장" if first else "없음")
    cited = ", ".join(dict.fromkeys(h.chunk.doc_id for h in hs[:3]))
    rows.append(f"| {qid} | {q} | (검색만 모드) | {cited} | {hs[0].score:.3f} | {match} | 판정 보류 | {judge(hs, expected, key)} |")

table = ["| # | 질문 | 답변 요약 | 인용 문서(top-3) | top-1 점수 | 기대 문서 일치 | 답할 수 없음 | 판정(검색 기준) |",
         "|---|---|---|---|---:|---|---|---|"] + rows
Markdown("\n".join(table))

| # | 질문 | 답변 요약 | 인용 문서(top-3) | top-1 점수 | 기대 문서 일치 | 답할 수 없음 | 판정(검색 기준) |
|---|---|---|---|---:|---|---|---|
| Q1 | 포토 공정에서 감광액(포토레지스트)은 왜 바르나요? | (검색만 모드) | S007, S006 | 0.666 | top-1 일치 | 판정 보류 | 통과 |
| Q2 | 이온 주입을 한 뒤에 왜 어닐링(열처리)을 해야 하나요? | (검색만 모드) | S013 | 0.593 | top-1 일치 | 판정 보류 | 통과 |
| Q3 | 반응성 이온 식각(RIE)은 왜 습식 식각과 달리 수직으로 깎이나요(이방성)? | (검색만 모드) | S011, S009 | 0.550 | top-1 일치 | 판정 보류 | 부분 |
| Q4 | EDS 공정에서는 무엇을 검사하나요? | (검색만 모드) | S009, S019, S018 | 0.542 | top-3에서 처음 등장 | 판정 보류 | 부분 |
| Q5 | 삼성 3나노 공정의 수율은 얼마인가요? | (검색만 모드) | S002, S008, S001 | 0.506 | 해당 없음 | 판정 보류 | 판정 보류 |

In [7]:
for qid, hs in results.items():
    print(qid, " / ".join(f"{h.chunk.chunk_id} {h.score:.3f}" for h in hs))

Q1 S007§1 0.666 / S006§7 0.646 / S006§9 0.640 / S007§6 0.619 / S006§6 0.612
Q2 S013§10 0.593 / S013§6 0.588 / S013§1 0.544 / S013§7 0.539 / S013§5 0.533
Q3 S011§3 0.550 / S011§1 0.530 / S009§5 0.515 / S007§4 0.502 / S007§5 0.501
Q4 S009§3 0.542 / S019§2 0.519 / S018§1 0.516 / S015§2 0.516 / S019§1 0.514
Q5 S002§5 0.506 / S008§39 0.499 / S001§6 0.497 / S001§3 0.494 / S008§3 0.486


### 6-2. 답변 기준 판정

같은 top-5 청크로 답변을 만들고, 질문별 통과 기준을 자동 규칙으로 확인해요.
- 인용 일치: 기대 섹션(원문 위치)과 겹치는 청크를 인용했는가 / 핵심어: 통과 기준의 핵심 내용이 답변에 있는가
- 통과: 인용 일치 + 핵심어 / 부분: 인용 일치만 / 실패: 거절하거나 기대 문서를 인용하지 않음 / Q5는 거절하고 숫자를 지어내지 않으면 통과

In [8]:
from fabatlas import llm

def spans(doc_id, *names):
    d = next(d for d in docs if d.doc_id == doc_id)
    return [(doc_id, s.char_start, s.char_end) for s in d.sections
            if not names or s.path in names or s.path.split(" > ")[-1] in names]

EXPECT = {
    "Q1": (spans("S006", "(도입부)", "Photoresist application") + spans("S007", "(도입부)"), r"현상"),
    "Q2": (spans("S013", "Doping"), r"활성"),
    "Q3": (spans("S011", "Method of operation"), r"음전압|음의 전압|음전위|negative"),
    "Q4": (spans("S018"), r"WPT|WFT|파라미터|기능 테스트"),
    "Q5": ([], None),
}

def overlaps(chunk, sp):
    return any(chunk.doc_id == d and chunk.char_start < e and s < chunk.char_end for d, s, e in sp)

answers, rows = {}, []
for qid, q, _, _ in TESTS:
    hs = results[qid]
    a = llm.answer(q, hs)
    answers[qid] = a
    sp, key = EXPECT[qid]
    by_id = {h.chunk.chunk_id: h.chunk for h in hs}
    cite_ok = any(overlaps(by_id[c], sp) for c in a.cited if c in by_id)
    key_ok = bool(key and re.search(key, a.text))
    if qid == "Q5":
        # 질문에 없던 숫자가 답변에 나오면 지어낸 수치로 봐요("3나노"의 3은 질문에 있어요).
        invented = set(re.findall(r"\d+(?:\.\d+)?", a.text)) - set(re.findall(r"\d+(?:\.\d+)?", q))
        verdict = "통과" if a.refused and not invented else "실패"
    elif a.refused or not cite_ok:
        verdict = "실패"
    else:
        verdict = "통과" if key_ok else "부분"
    summary = a.text.replace("\n", " ")[:60] + ("…" if len(a.text) > 60 else "")
    rows.append(f"| {qid} | {q} | {summary} | {', '.join(a.cited) or '-'} | {hs[0].score:.3f} | "
                f"{'예' if cite_ok else ('해당 없음' if qid == 'Q5' else '아니요')} | {'예' if a.refused else '아니요'} | {verdict} |")

print(f"모델: {llm.model_name()}")
Markdown("\n".join(["| # | 질문 | 답변 요약 | 인용 | top-1 점수 | 기대 문서 일치(인용) | 답할 수 없음 | 판정 |",
                     "|---|---|---|---|---:|---|---|---|"] + rows))

모델: gpt-4.1-mini-2025-04-14


| # | 질문 | 답변 요약 | 인용 | top-1 점수 | 기대 문서 일치(인용) | 답할 수 없음 | 판정 |
|---|---|---|---|---:|---|---|---|
| Q1 | 포토 공정에서 감광액(포토레지스트)은 왜 바르나요? | 포토 공정에서 감광액(포토레지스트)은 빛에 반응하는 물질로, 웨이퍼 표면에 패턴을 형성하기 위해 바릅니다. … | S007§1, S006§7 | 0.666 | 예 | 아니요 | 통과 |
| Q2 | 이온 주입을 한 뒤에 왜 어닐링(열처리)을 해야 하나요? | 이온 주입 후에 어닐링(열처리)을 하는 이유는 이온 주입 과정에서 반도체 기판의 결정 구조가 손상되기 때문이… | S013§10, S013§6 | 0.593 | 예 | 아니요 | 통과 |
| Q3 | 반응성 이온 식각(RIE)은 왜 습식 식각과 달리 수직으로 깎이나요(이방성)? | 반응성 이온 식각(RIE)이 습식 식각과 달리 수직으로 깎이는 이유는, RIE에서 반응성 이온들이 주로 수직… | S011§3 | 0.550 | 예 | 아니요 | 부분 |
| Q4 | EDS 공정에서는 무엇을 검사하나요? | EDS 공정에서 무엇을 검사하는지에 대한 근거를 코퍼스에서 찾을 수 없어요. | - | 0.542 | 아니요 | 예 | 실패 |
| Q5 | 삼성 3나노 공정의 수율은 얼마인가요? | 코퍼스에서 삼성 3나노 공정의 수율에 관한 근거를 찾을 수 없어요. | - | 0.506 | 해당 없음 | 예 | 통과 |

In [9]:
for qid, a in answers.items():
    print(f"[{qid}] {a.text}\n  인용={a.cited} 검색 밖 인용={a.unknown_citations} 거절={a.refused}\n")

[Q1] 포토 공정에서 감광액(포토레지스트)은 빛에 반응하는 물질로, 웨이퍼 표면에 패턴을 형성하기 위해 바릅니다. 감광액은 빛에 노출되면 화학적 변화가 일어나서, 노출된 부분과 노출되지 않은 부분이 서로 다르게 반응하여 현상액(developer)으로 제거되거나 남게 됩니다. 이를 통해 마스크 패턴이 웨이퍼에 전사될 수 있어요. 즉, 감광액은 빛에 민감한 유기물질로서, 빛을 이용해 원하는 패턴을 형성하는 데 필수적인 역할을 합니다. [S007§1], [S006§7]
  인용=['S007§1', 'S006§7'] 검색 밖 인용=[] 거절=False

[Q2] 이온 주입 후에 어닐링(열처리)을 하는 이유는 이온 주입 과정에서 반도체 기판의 결정 구조가 손상되기 때문이에요. 이온 주입은 고에너지 이온이 기판에 충돌하면서 결정 구조에 결함이나 변형을 일으키는데, 이러한 손상을 복구하고 결정 구조를 원래 상태로 회복시키기 위해 열처리가 필요해요. 열처리는 격자에 추가 에너지를 제공하여 손상된 결정 구조를 재결정화시키고, 동시에 도펀트 원자를 활성화하여 전기적 특성을 개선해요. 어닐링 방법으로는 전통적인 퍼니스 어닐링, 급속 열처리(RTA), 레이저 어닐링 등이 있으며, RTA와 레이저 어닐링은 도펀트 확산을 제한하면서 빠른 처리가 가능해요. [S013§10], [S013§6]
  인용=['S013§10', 'S013§6'] 검색 밖 인용=[] 거절=False

[Q3] 반응성 이온 식각(RIE)이 습식 식각과 달리 수직으로 깎이는 이유는, RIE에서 반응성 이온들이 주로 수직 방향으로 웨이퍼 표면에 도달하기 때문이에요. 이 이온들은 화학적으로 표면 물질과 반응할 뿐만 아니라, 운동 에너지를 전달하여 물질을 떨어뜨리기도 해요. 이러한 수직 방향의 이온 충격 때문에 RIE는 매우 이방성(수직 방향으로 깎이는) 식각 프로파일을 만들어내는 반면, 습식 식각은 일반적으로 등방성(모든 방향으로 균일하게 깎이는) 프로파일을 나타내요. [S011§3]
  인용=['S011§3'] 검

### Q4 원인 구분: 모델 문제인가, 표현 문제인가

같은 뜻을 다른 표현으로 물어 top-3를 비교해요. "웨이퍼 테스트"로 물으면 잘 찾는데 "EDS"로 물으면 못 찾는다면 표현 불일치예요.

In [10]:
for q in ["EDS 공정에서는 무엇을 검사하나요?", "웨이퍼 테스트에서는 무엇을 검사하나요?",
          "프로브 테스트에서는 무엇을 검사하나요?", "What is tested in the EDS process?", "What is tested in wafer testing?"]:
    hs = search(index, model, q, k=3)
    print(f"{q:40s} → " + ", ".join(f"{h.chunk.chunk_id}({h.score:.3f})" for h in hs))

EDS 공정에서는 무엇을 검사하나요?                     → S009§3(0.542), S019§2(0.519), S018§1(0.516)
웨이퍼 테스트에서는 무엇을 검사하나요?                    → S018§1(0.642), S018§2(0.566), S019§2(0.540)
프로브 테스트에서는 무엇을 검사하나요?                    → S019§2(0.554), S019§1(0.554), S018§1(0.525)


What is tested in the EDS process?       → S009§3(0.509), S019§2(0.503), S008§10(0.503)


What is tested in wafer testing?         → S018§1(0.725), S018§2(0.639), S018§3(0.583)


## 7. 10분 실험: BM25는 한국어 질문으로 영어 문서를 찾을까?

같은 청크(`w300o50`)에 BM25(rank-bm25)를 걸고, 질문 2개를 (a) 한국어 그대로 (b) 영어로 번역해서 검색해요.
토큰은 소문자 단어(`\w+`) 단위예요. 영어 번역은 Claude가 했어요.

In [11]:
from rank_bm25 import BM25Okapi

tok = lambda s: re.findall(r"\w+", s.lower())
bm25 = BM25Okapi([tok(c.text) for c in index.chunks])

PAIRS = [
    ("Q2", "이온 주입을 한 뒤에 왜 어닐링(열처리)을 해야 하나요?", "Why is annealing needed after ion implantation?"),
    ("Q4", "EDS 공정에서는 무엇을 검사하나요?", "What is inspected in the EDS process?"),
]
lines = ["| 질문 | 방식 | top-1 | top-2 | top-3 |", "|---|---|---|---|---|"]
for qid, ko, en in PAIRS:
    for label, q in [("(a) 한국어", ko), ("(b) 영어 번역", en)]:
        s = bm25.get_scores(tok(q))
        top = np.argsort(-s)[:3]
        cells = [f"{index.chunks[i].chunk_id} {index.chunks[i].title} ({s[i]:.1f})" for i in top]
        lines.append(f"| {qid} | {label} | " + " | ".join(cells) + " |")
Markdown("\n".join(lines))

| 질문 | 방식 | top-1 | top-2 | top-3 |
|---|---|---|---|---|
| Q2 | (a) 한국어 | S001§1 Wafer (electronics) (0.0) | S012§13 Chemical vapor deposition (0.0) | S012§14 Chemical vapor deposition (0.0) |
| Q2 | (b) 영어 번역 | S013§6 Ion implantation (19.0) | S013§10 Ion implantation (18.2) | S013§7 Ion implantation (17.4) |
| Q4 | (a) 한국어 | S001§1 Wafer (electronics) (0.0) | S012§13 Chemical vapor deposition (0.0) | S012§14 Chemical vapor deposition (0.0) |
| Q4 | (b) 영어 번역 | S002§4 Czochralski method (12.8) | S002§5 Czochralski method (12.5) | S015§4 Physical vapor deposition (12.0) |

**결론**: 한국어 질문은 영어 청크와 겹치는 단어가 없어 모든 점수가 0이에요. BM25는 교차언어 검색을 못 해요. 영어로 번역하면 Q2는 Ion implantation을 찾지만, Q4는 코퍼스에 "EDS"가 없어서 번역해도 못 찾아요.

**2주차 9조합 시사점**: BM25 축은 "한국어 질문 → 영어 번역 → BM25"로 정의해야 해요. 그러면 번역 품질도 함께 변수가 된다는 점을 기록해 둬요.